# Μάθημα 10 - Πράκτορες AI σε Παραγωγή

Σε αυτό το μάθημα θα μάθετε **πρότυπα παραγωγής** για πράκτορες AI χρησιμοποιώντας το Microsoft Agent Framework (Python). Καλύπτουμε:

- **Παρατηρησιμότητα** — προσθήκη χρονισμού και καταγραφής στις αλληλεπιδράσεις του πράκτορα
- **Αξιολόγηση** — χρήση ενός πράκτορα αξιολογητή για να βαθμολογεί την ποιότητα των απαντήσεων
- **Διαχείριση κόστους** — στρατηγικές για βελτιστοποίηση των token και επιλογή μοντέλου

Το σενάριο είναι ένας **ταξιδιωτικός πράκτορας** που βοηθά τους χρήστες να οργανώνουν ταξίδια, με ενσωματωμένη παρακολούθηση και αξιολόγηση.


## Ρύθμιση


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Θέματα για την παραγωγή

Η μετάβαση των πρακτόρων τεχνητής νοημοσύνης από πρωτότυπα σε παραγωγή απαιτεί προσεκτική προσοχή σε τρεις πυλώνες:

1. **Παρατηρησιμότητα** — Χρειάζεστε ορατότητα σε αυτό που κάνει ο πράκτορας, πόσο διαρκεί και ποια εργαλεία καλεί. Χωρίς ιχνηλάτηση και καταγραφή, η αποσφαλμάτωση προβλημάτων στην παραγωγή είναι σχεδόν αδύνατη.

2. **Αξιολόγηση** — Οι αυτοματοποιημένοι έλεγχοι ποιότητας διασφαλίζουν ότι οι απαντήσεις του πράκτορα παραμένουν ακριβείς, πλήρεις και χρήσιμες με την πάροδο του χρόνου. Ένας αξιολογητής πράκτορας μπορεί να βαθμολογήσει τις απαντήσεις με βάση προκαθορισμένα κριτήρια.

3. **Διαχείριση Κόστους** — Η χρήση tokens επηρεάζει άμεσα το κόστος. Στρατηγικές όπως η βελτιστοποίηση προτροπών, η επιλογή μοντέλου και η προσωρινή αποθήκευση (caching) βοηθούν να διατηρηθούν τα έξοδα υπό έλεγχο χωρίς να θυσιάζεται η ποιότητα.


## Δημιουργία ενός Παρατηρήσιμου Πράκτορα

Ορίζουμε εργαλεία ταξιδιού και τυλίγουμε την κλήση του πράκτορα με χρονισμό ώστε να μπορούμε να παρακολουθούμε την καθυστέρηση. Σε περιβάλλον παραγωγής θα ενσωματώνατε το OpenTelemetry ή παρόμοιο σύστημα ιχνηλάτησης.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Πρότυπα Αξιολόγησης

Ένα κοινό παραγωγικό πρότυπο είναι να χρησιμοποιείται ένας δεύτερος πράκτορας ως **αξιολογητής**. Ο αξιολογητής βαθμολογεί την απάντηση του κύριου πράκτορα με βάση προκαθορισμένα κριτήρια όπως η πληρότητα, η ακρίβεια και η χρησιμότητα.

Αυτό επιτρέπει:
- Αυτοματοποιημένους ποιοτικούς ελέγχους πριν οι απαντήσεις φτάσουν στους χρήστες
- Ανίχνευση παλινδρόμησης όταν αλλάζουν οι προτροπές ή τα μοντέλα
- Συνεχή παρακολούθηση της απόδοσης του πράκτορα με την πάροδο του χρόνου


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Στρατηγικές Διαχείρισης Κόστους

Ο έλεγχος του κόστους είναι κρίσιμος για παραγωγικούς πράκτορες AI. Εδώ είναι βασικές στρατηγικές:

| Strategy | Description |
|---|---|
| **Βελτιστοποίηση προτροπών** | Κρατήστε τις εντολές συστήματος σύντομες. Αφαιρέστε περιττό πλαίσιο για να μειώσετε τα εισαγόμενα tokens. |
| **Επιλογή μοντέλου** | Χρησιμοποιήστε μικρότερα, φθηνότερα μοντέλα (π.χ. GPT-4o-mini) για απλά καθήκοντα όπως ταξινόμηση ή εξαγωγή, και κρατήστε τα μεγαλύτερα μοντέλα για σύνθετη συλλογιστική. |
| **Προσωρινή μνήμη (Caching)** | Αποθηκεύστε αποτελέσματα εργαλείων και συχνές ερωτήσεις στην cache για να αποφύγετε περιττές κλήσεις API. |
| **Όρια token** | Ορίστε όρια `max_tokens` για να αποτρέψετε απρόσμενα μακρές απαντήσεις. |
| **Ομαδοποίηση (Batching)** | Συγκεντρώστε πολλαπλές αιτήσεις χρήστη σε μία κλήση API όπου είναι δυνατό. |

Στην πράξη, μία βαθμιδωτή προσέγγιση λειτουργεί καλά: κατευθύνετε απλές αιτήσεις σε ένα γρήγορο, οικονομικό μοντέλο και αναβαθμίζετε μόνο τις σύνθετες ερωτήσεις σε ένα πιο ικανό.


## Περίληψη

Σε αυτό το μάθημα μάθατε πώς:

1. **Προσθήκη παρατηρησιμότητας** στις αλληλεπιδράσεις του πράκτορα με χρονομέτρηση και καταγραφή, θέτοντας τα θεμέλια για ιχνηλάτηση και παρακολούθηση.
2. **Αξιολόγηση των απαντήσεων του πράκτορα** αυτόματα χρησιμοποιώντας έναν αξιολογητή πράκτορα που βαθμολογεί την πληρότητα, την ακρίβεια και τη χρησιμότητα.
3. **Διαχείριση κόστους** μέσω βελτιστοποίησης προτροπών, επιλογής μοντέλου, caching, και προϋπολογισμών tokens.

Αυτά τα πρότυπα παραγωγής βοηθούν να διασφαλιστεί ότι οι πράκτορές σας τεχνητής νοημοσύνης είναι αξιόπιστοι, μετρήσιμοι και οικονομικά αποδοτικοί σε μεγάλη κλίμακα.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Αποποίηση ευθυνών:

Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία αυτόματης μετάφρασης AI Co‑op Translator (https://github.com/Azure/co-op-translator). Παρόλο που επιδιώκουμε την ακρίβεια, παρακαλούμε λάβετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν σφάλματα ή ανακρίβειες. Το πρωτότυπο έγγραφο στη γλώσσα στην οποία συντάχθηκε πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες συνιστάται επαγγελματική μετάφραση από επαγγελματία μεταφραστή. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή παρανοήσεις που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
